In [25]:
import pandas as pd
import requests
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text 

In [30]:
def get_legislatures(id): 
    # URL for legislatures in Brazil
    url = f"https://dadosabertos.camara.leg.br/api/v2/deputados?idLegislatura={id}&ordem=ASC&ordenarPor=nome"

    # Get the legislatures
    try:
        response = requests.get(url)

        if response.status_code != 200:
            raise Exception
        else:
            data = response.json()

            # Transform to pandas Dataframe
            df = pd.DataFrame(data["dados"])


    except Exception as e:
        print("Error:", e)

    return df

In [ ]:
def load_supabase(table_name, schema, mode="replace"):
    database_url = os.getenv("DATABASE_CONNECTION")

    # Create Engine
    engine = create_engine(database_url)

    # Create Schema if not exists
    with engine.connect() as conn:
        conn.execute(text("create schema if not exists bronze"))
        conn.commit()

    # Read Parquet file
    df = pd.read_parquet(f"data/raw/{table_name}/{table_name}.parquet")

    # Save to Bronze laywer
    df.to_sql(
        name= {table_name},
        con= engine,
        schema= schema,
        if_exists= mode,
        index=False
    )

In [40]:
dfs = []
INITIAL_LEGISLATURE = 54
FINAL_LEGISLATURE = 57

for legislature in range(INITIAL_LEGISLATURE, FINAL_LEGISLATURE + 1):

    # Extract Data
    data = get_legislatures(legislature)

    # Append
    dfs.append(data)

# Concatennate dataframens
df = pd.concat(dfs, ignore_index=True)


In [41]:
display(df)

,id,uri,nome,siglaPartido,uriPartido,siglaUf,idLegislatura,urlFoto,email
0,141463,https://dadosabertos.camara.leg.br/api/v2/depu...,ABELARDO CAMARINHA,PSB,https://dadosabertos.camara.leg.br/api/v2/part...,SP,54,https://www.camara.leg.br/internet/deputado/ba...,None
1,73764,https://dadosabertos.camara.leg.br/api/v2/depu...,ABELARDO LUPION,DEM,https://dadosabertos.camara.leg.br/api/v2/part...,PR,54,https://www.camara.leg.br/internet/deputado/ba...,None
2,161907,https://dadosabertos.camara.leg.br/api/v2/depu...,ACELINO POPÓ,PRB,https://dadosabertos.camara.leg.br/api/v2/part...,BA,54,https://www.camara.leg.br/internet/deputado/ba...,None
3,133374,https://dadosabertos.camara.leg.br/api/v2/depu...,ADEMIR CAMILO,PDT,https://dadosabertos.camara.leg.br/api/v2/part...,MG,54,https://www.camara.leg.br/internet/deputado/ba...,None
4,133374,https://dadosabertos.camara.leg.br/api/v2/depu...,ADEMIR CAMILO,PSD,https://dadosabertos.camara.leg.br/api/v2/part...,MG,54,https://www.camara.leg.br/internet/deputado/ba...,None
...,...,...,...,...,...,...,...,...,...
3733,204517,https://dadosabertos.camara.leg.br/api/v2/depu...,Zé Vitor,PL,https://dadosabertos.camara.leg.br/api/v2/part...,MG,57,https://www.camara.leg.br/internet/deputado/ba...,dep.zevitor@camara.leg.br
3734,160592,https://dadosabertos.camara.leg.br/api/v2/depu...,Zeca Dirceu,PT,https://dadosabertos.camara.leg.br/api/v2/part...,PR,57,https://www.camara.leg.br/internet/deputado/ba...,dep.zecadirceu@camara.leg.br
3735,220592,https://dadosabertos.camara.leg.br/api/v2/depu...,Zezinho Barbary,PP,https://dadosabertos.camara.leg.br/api/v2/part...,AC,57,https://www.camara.leg.br/internet/deputado/ba...,dep.zezinhobarbary@camara.leg.br
3736,220552,https://dadosabertos.camara.leg.br/api/v2/depu...,Zucco,REPUBLICANOS,https://dadosabertos.camara.leg.br/api/v2/part...,RS,57,https://www.camara.leg.br/internet/deputado/ba...,dep.zucco@camara.leg.br


In [45]:
# Save Parquet =
df.to_parquet(
    f"../../data/raw/legislatures/legislatures.parquet",
    index= False
)